# Detecção de Fraudes em Cartões de Crédito

**Objetivo:** Construir um pipeline completo de Machine Learning para detectar transações fraudulentas em dados altamente desbalanceados (0,17% de fraudes).

**Foco:** Maximizar o **Recall** da classe de fraude ($Class = 1$) sem comprometer excessivamente a **Precisão**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score, recall_score, precision_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
import shap

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

## 2. Carregamento e Análise Exploratória (EDA)

In [ ]:
url = "https://raw.githubusercontent.com/datasets/credit-card-fraud/main/data/creditcard.csv"
df = pd.read_csv(url)

print(f"Dimensões do Dataset: {df.shape[0]} linhas e {df.shape[1]} colunas\n")

classes = df['Class'].value_counts(normalize=True) * 100
print(f"Transações Legítimas (0): {classes[0]:.2f}%")
print(f"Transações Fraudulentas (1): {classes[1]:.2f}%\n")

plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Class', data=df, palette='viridis')
plt.title('Distribuição de Classes (0: Normal | 1: Fraude)')
plt.show()

## 3. Pré-processamento e Engenharia de Atributos

In [ ]:
df['Amount_Log'] = np.log1p(df['Amount'])

scaler = StandardScaler()
df['Scaled_Time'] = scaler.fit_transform(df[['Time']])
df['Scaled_Amount'] = scaler.fit_transform(df[['Amount_Log']])

X = df.drop(columns=['Class', 'Time', 'Amount', 'Amount_Log'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Base de Treino: {X_train.shape[0]} amostras | Fraudes: {y_train.sum()}")
print(f"Base de Teste:  {X_test.shape[0]} amostras | Fraudes: {y_test.sum()}")

## 4. Comparação de Algoritmos e Pesos de Classe

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)
xgb = XGBClassifier(scale_pos_weight=scale_weight, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

modelos = {'Regressão Logística': lr, 'Random Forest': rf, 'XGBoost': xgb}
resultados = []

for nome, mod in modelos.items():
    preds = mod.predict(X_test)
    resultados.append({
        'Modelo': nome,
        'Recall (Fraude)': recall_score(y_test, preds, pos_label=1),
        'Precisão (Fraude)': precision_score(y_test, preds, pos_label=1),
        'F1-Score (Fraude)': f1_score(y_test, preds, pos_label=1)
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

## 5. Otimização do Limiar de Decisão (Threshold Tuning)

In [ ]:
y_probs_xgb = xgb.predict_proba(X_test)[:, 1]
custom_threshold = 0.30
y_pred_custom = (y_probs_xgb >= custom_threshold).astype(int)

print(f"--- Relatório de Classificação (XGBoost | Threshold = {custom_threshold}) ---")
print(classification_report(y_test, y_pred_custom, target_names=['Legítima', 'Fraude']))

## 6. Avaliação Gráfica das Métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_custom)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legítima', 'Fraude'], yticklabels=['Legítima', 'Fraude'])
axes[0].set_title(f'Matriz de Confusão (Threshold = {custom_threshold})')
axes[0].set_xlabel('Previsto')
axes[0].set_ylabel('Real')

precision, recall, thresholds = precision_recall_curve(y_test, y_probs_xgb)
axes[1].plot(recall, precision, color='purple', lw=2, label='XGBoost')
axes[1].set_xlabel('Recall (Sensibilidade)')
axes[1].set_ylabel('Precisão')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Explicabilidade do Modelo com SHAP

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)